# Weekly redevelopment: separate intended use

Revised 2026-09-22.

Original quarterly implementations were reviewed first. Weekly redevelopment changes the forecast target/horizon, so it is a separate model version. These results do not establish failure of a correctly implemented quarterly strategy. Frozen weekly choices and CLAM weights are unchanged.


In [1]:
from pathlib import Path
import sys, pandas as pd
ROOT = Path.cwd()
if ROOT.name == "notebooks": ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
import config
from sv import db
con = db.connect(read_only=True)


In [2]:
pd.read_csv(ROOT/'reports/oot_performance.csv')

,model,weeks,CAGR,vol,Sharpe,max DD,active SR
0,momentum,241,37.8%,41.1%,0.986826,-28.6%,0.897571
1,gbm,241,10.5%,31.5%,0.474340,-37.1%,0.064542
2,gbm_expected,241,18.4%,41.3%,0.613995,-34.5%,0.391066
3,gbm_weekly,241,22.9%,41.2%,0.706149,-39.6%,0.500796
4,clam_2021,241,17.2%,34.2%,0.634844,-32.4%,0.359972
5,clam_weekly_cs_demeaned,241,-4.1%,29.7%,0.009879,-50.6%,-0.894509
6,clam_weekly_cs_rank_small,241,10.3%,24.7%,0.521729,-27.8%,-0.089091
7,clam_weekly_cs_rank_small_n94_seed20260922,241,9.2%,37.4%,0.422693,-49.8%,0.087179
8,clam_weekly_cs_rank_small_n500_seed20260922,241,11.0%,24.2%,0.552052,-32.8%,-0.040217
9,clam_weekly_cs_rank_small_n3000_seed20260922,241,-0.3%,15.3%,0.057856,-25.5%,-0.987650


In [3]:
pd.read_csv(ROOT/'reports/oot_tests.csv')

,model,active SR [95% CI],bootstrap,permutation p,permutation,DSR (provisional)
0,momentum,"0.90 [0.26, 1.53]",PASS,0.019960,PASS,0.390373
1,gbm,"0.06 [-0.80, 0.86]",FAIL,0.083832,FAIL,0.019312
2,gbm_expected,"0.39 [-0.31, 1.12]",FAIL,0.163673,FAIL,0.085871
3,gbm_weekly,"0.50 [-0.12, 1.15]",FAIL,0.105788,FAIL,0.128904
4,clam_2021,"0.36 [-0.37, 1.05]",FAIL,0.135729,FAIL,0.075031
5,clam_weekly_cs_demeaned,"-0.89 [-1.56, -0.19]",FAIL,0.838323,FAIL,0.000016
6,clam_weekly_cs_rank_small,"-0.09 [-0.82, 0.65]",FAIL,0.323353,FAIL,0.008286
7,clam_weekly_cs_rank_small_n94_seed20260922,"0.09 [-0.83, 1.01]",FAIL,0.307385,FAIL,0.021661
8,clam_weekly_cs_rank_small_n500_seed20260922,"-0.04 [-0.80, 0.75]",FAIL,0.267465,FAIL,0.010894
9,clam_weekly_cs_rank_small_n3000_seed20260922,"-0.99 [-1.65, -0.35]",FAIL,0.902196,FAIL,0.000008


In [4]:
import json
pd.DataFrame(json.loads((ROOT/'experiments.json').read_text())['trials'])

,model,family,status,seed,training_cutoff,evaluation_cutoff,selection,changes
0,momentum,baseline,available,NaN,NaN,NaN,NaN,NaN
1,gbm,baseline,available,NaN,NaN,NaN,NaN,NaN
2,gbm_expected,baseline,available,NaN,NaN,NaN,NaN,NaN
3,gbm_w_63_expected,weekly redevelopment,available,NaN,NaN,NaN,NaN,NaN
4,gbm_w_63_prob_up,weekly redevelopment,available,NaN,NaN,NaN,NaN,NaN
5,gbm_w_126_expected,weekly redevelopment,available,NaN,NaN,NaN,NaN,NaN
6,gbm_w_126_prob_up,weekly redevelopment,available,NaN,NaN,NaN,NaN,NaN
7,gbm_w_252_expected,weekly redevelopment,available,NaN,NaN,NaN,NaN,NaN
8,gbm_w_252_prob_up,weekly redevelopment,available,NaN,NaN,NaN,NaN,NaN
9,clam_weekly_raw,weekly redevelopment,missing historical scores,NaN,NaN,NaN,NaN,NaN


## Training-universe expansion

New paired top-500 and top-3000 runs use the same small rank-target architecture, seed and purged validation split. Original artifacts are preserved. OOT has been reused; the paired test is diagnostic rather than fresh confirmation.

In [5]:
pd.read_csv(ROOT/'reports/clam_universe_comparison.csv')

,model,requested_tickers,usable_tickers,training_windows,validation_windows,purged_windows,epochs,validation_rank_ic,oot_weeks,oot_cagr,oot_sharpe,oot_active_sharpe,oot_max_drawdown,bootstrap,permutation_p
0,clam_weekly_cs_rank_small_n94_seed20260922,94,89,27869,8790,87,8,0.066983,241,0.092252,0.422693,0.087179,-0.498342,FAIL,0.307385
1,clam_weekly_cs_rank_small_n500_seed20260922,500,476,142185,46633,459,7,0.023968,241,0.109698,0.552052,-0.040217,-0.328301,FAIL,0.267465
2,clam_weekly_cs_rank_small_n3000_seed20260922,3000,2567,659736,237750,2263,9,0.016005,241,-0.002933,0.057856,-0.987650,-0.255018,FAIL,0.902196


In [6]:
json.loads((ROOT/'reports/clam_universe_paired_test.json').read_text())

{'control': 500, 'block_weeks': 13, 'draws': 5000, 'contrasts': [{'contrast': '94 minus matched 500 control; net weekly portfolio returns', 'annualized_mean_return_difference': 0.024826198821483587, 'ci_95': [-0.14557377392873194, 0.2177991882911551], 'weeks': 241}, {'contrast': '3000 minus matched 500 control; net weekly portfolio returns', 'annualized_mean_return_difference': -0.12461161184178418, 'ci_95': [-0.23062508132318477, -0.012034570600388577], 'weeks': 241}], 'limitation': 'Single seed, reused OOT and later-universe snapshot. Not a general conclusion about training-set size.'}

In [7]:
con.close()